# 05 — N-grams & Collocations

**Learning objective.** Model local word order with contiguous token sequences and inspect which phrases carry signal.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
texts=['new york bank account','new york city travel','bank account fee','account fee refund']
vec=CountVectorizer(ngram_range=(1,2))
X=vec.fit_transform(texts)
features=np.array(vec.get_feature_names_out())
counts=np.asarray(X.sum(axis=0)).ravel()
rank=pd.DataFrame({'ngram':features,'count':counts}).sort_values(['count','ngram'],ascending=[False,True])
rank.head(12)

           ngram  count
0        account      3
1    account fee      2
2           bank      2
3   bank account      2
6            fee      2
8            new      2
9       new york      2
12          york      2
4           city      1
5    city travel      1
7     fee refund      1
10        refund      1

Bigram features can distinguish `credit card` from independent occurrences of `credit` and `card`, but the feature space grows rapidly. Character n-grams are especially useful for misspellings, morphology and language identification.

In [3]:
char=CountVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1)
char.fit(['colour','color','colours'])
print('character n-gram features:', len(char.get_feature_names_out()))
print(char.get_feature_names_out()[:20])

character n-gram features: 27
[' co' ' col' ' colo' 'col' 'colo' 'color' 'colou' 'lor' 'lor ' 'lou'
 'lour' 'lour ' 'lours' 'olo' 'olor' 'olor ' 'olou' 'olour' 'or ' 'our']


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain the sparsity/expressiveness trade-off of n-grams
- Use character n-grams for robust lexical matching